# CROSS · Analysis overview

按编号顺序执行。对比表与区间来自已验证回执；预览写入 results/_views/，不新增统计。


In [ ]:
import os
from pathlib import Path
from IPython.display import display, Image
from sglib.core.infra.paths import find_repo_root
from sglib.experiment.downstream import country_status, load_country_tables
from sglib.analysis import stage
from sglib.analysis.overview import tables
from sglib.analysis.manifest import run_gate, write_setup

REPO = find_repo_root(Path.cwd())
COUNTRY = 'cross'
PROFILE = os.environ.get("SG_PROFILE", "formal")
RESULTS_ROOT = os.environ.get("SG_RESULTS_ROOT")
UPSTREAM = {"status": country_status, "tables": load_country_tables}
OPTIONS = dict(profile=PROFILE, results_root=RESULTS_ROOT, upstream=UPSTREAM)
ctx = stage.country_context(REPO, COUNTRY, **OPTIONS)
display(tables.status(ctx))


## 综合表（Synthesis tables）

国家保持独立，逐表展示已有证据与适用边界。


In [ ]:
for name, table in tables.synthesis(ctx).items():
    print(f"{name}: {len(table)} rows")
    display(table)


## 覆盖审计与叶清单门（Coverage audit and leaf manifest gate）

封存根的首次审计只生成到视图目录。审计通过并落根之后才能生成清单，随后核对每个叶节点。


In [ ]:
stage.run_step(REPO, COUNTRY, ["audit"], **OPTIONS)
write_setup(ctx)
report = run_gate(ctx)
if report["status"] == "FAIL" or report["unclaimed"]:
    raise RuntimeError("Analysis manifest gate failed")
if (ctx.root / "audit.json").is_file():
    display(tables.audit(ctx))
